In [6]:
import numpy as np
from scipy.io import loadmat, savemat

# Load original Felix from MATLAB
mat = loadmat('/Users/mczhang/Documents/GitHub/FM5_ML/02-data/D_man/D_CC1_CCPo.mat', struct_as_record=False, squeeze_me=True)
Felix = mat['Felix']

# Load predictions (should match filtered valid waveforms)
X = np.load('your_data.npy')
y = np.load('your_labels.npy')

# Build and predict
from eqpolarity.utils import construct_model
model = construct_model((600, 1))
model.load_weights('models/best_weigths_Binary_SCSN_Best.h5')
y_pred_prob = model.predict(X)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

# Now: assign prediction to Felix.CC1_eqpPo
# Felix is 1D numpy array of MATLAB structs

valid_idx = []  # Indices where W1_CC1 exists and CCPo is ±1
for i, f in enumerate(Felix):
    if hasattr(f, 'W1_CC1') and isinstance(f.W1_CC1, np.ndarray) and len(f.W1_CC1) == 128:
        #if hasattr(f, 'CC1_CCPo') and np.abs(f.CC1_CCPo) == 3:
        valid_idx.append(i)

# Sanity check
if len(valid_idx) != len(y_pred):
    raise ValueError(f"Mismatch between prediction and valid waveforms: {len(valid_idx)} valid indices vs {len(y_pred)} predictions")

# Assign prediction
for k, i in enumerate(valid_idx):
    setattr(Felix[i], 'CC1_eqpPo', int(y_pred[k]))  # Save 0 or 1
# Save modified Felix struct back to MATLAB
savemat('Felix_with_eqpPo.mat', {'Felix': Felix})


32/32 [==============================] - 18s 541ms/step
